# 096 — Proyecto: pipeline creativo trazable

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

Un **pipeline creativo trazable** encadena etapas de generación (texto → imagen →
edición…) donde cada etapa produce un **contrato JSON** con: modelo y versión, semilla,
prompt/parámetros, `input_hash` (hash de la salida de la etapa anterior) y
`output_hash` (hash de la salida propia). Tres decisiones de diseño:

1. **Determinismo declarado** — modelo + versión + semilla + parámetros registrados:
   auditabilidad aunque no haya reproducción bit a bit entre hardwares.
2. **Hashes encadenados** — `input_hash(n) == output_hash(n−1)`; el manifiesto final
   encadena el linaje completo, al estilo de la cadena de manifiestos C2PA.
3. **Registro append-only** — los manifiestos no se editan; re-ejecutar crea entradas
   nuevas. Un manifiesto reconstruido a posteriori no es evidencia.

**Evaluación**: utilidad (¿cumple la especificación creativa?), autenticidad
verificable (la cadena de hashes cierra; al publicar, el manifiesto viaja con el
activo — C2PA/Content Credentials) y gobernanza (consentimiento y licencias de cada
insumo, riesgo gestionado al estilo NIST AI RMF: identificar–medir–gestionar).

## 🧮 Ejemplo de referencia

Hash didáctico de 8 bits: `h(s) = (Σ ord(c)) mod 256`. Pipeline de 3 etapas:

```text
s1 = "un faro al amanecer"                 → h(s1) = 0xF4
s2 = "IMG[faro,amanecer,seed=7]"           → h(s2) = 0xE6   (input_hash = F4 ✔)
s3 = "IMG[faro,amanecer,seed=7]+recorte"   → h(s3) = 0x05   (input_hash = E6 ✔)
cadena F4 → E6 → 05: VERIFICA
```

Si alguien regenera la etapa 2 con `seed=8` sin registrarlo, h(s2') = 0xE7 ≠ 0xE6 y la
verificación falla en el eslabón 2→3: la cadena detecta la alteración, no la repara.
Con SHA-256 real, fabricar una s2' con el mismo hash es computacionalmente inviable.

In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("capstone", seed=96)
show(result)


## Reflexión

1. Con la misma semilla pero una versión nueva del modelo en la etapa 2, la salida cambia y la cadena de hashes rompe. ¿Qué campo del contrato explica la diferencia y por qué semilla sin versión no basta para auditar?
2. ¿Por qué un manifiesto reconstruido después de la publicación no constituye evidencia de procedencia aunque todos los hashes cuadren?
3. ¿Qué añade una firma criptográfica (C2PA) sobre la cadena de hashes simple de este proyecto, y qué problema operativo nuevo introduce?